<a href="https://colab.research.google.com/github/shubhangiisankar/AICTE-Internships-2025-June_2025-Water-Quality-Prediction/blob/main/Copy_of_DrugDrugInteraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers datasets accelerate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer
)


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive

'Colab Notebooks'  'drug_with_unique_side_effects (1).xlsx'


In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/drug_with_unique_side_effects (1).xlsx"

original_df = pd.read_excel(data_path)

original_df.columns = original_df.columns.str.strip().str.lower()
df = original_df.copy()

print(df.columns)
df.head()


Index(['drugbank_id', 'name', 'description', 'type', 'side_effects'], dtype='object')


,drugbank_id,name,description,type,side_effects
0,DB00001,Lepirudin,Lepirudin is identical to natural hirudin exce...,biotech,"Back pain, Sore throat, Cough, Tingling, Depre..."
1,DB00002,Cetuximab,Cetuximab is an epidermal growth factor recept...,biotech,"Muscle pain, Chills, Nausea, Low blood pressur..."
2,DB00003,Dornase alfa,Dornase alfa is a biosynthetic form of human d...,biotech,"Abdominal pain, Numbness, Photosensitivity, Fe..."
3,DB00004,Denileukin diftitox,A recombinant DNA-derived cytotoxic protein co...,biotech,"Fatigue, Weakness, Kidney dysfunction, Diarrhe..."
4,DB00005,Etanercept,Dimeric fusion protein consisting of the extra...,biotech,"Headache, Hives, Weakness, Weight gain, Shortn..."


In [ ]:
df["text"] = (
    "Drug name: " + df["name"].astype(str) + ". "
    "Description: " + df["description"].astype(str) + ". "
    "Known side effects include: " + df["side_effects"].astype(str) + "."
)

# Dummy label for representation learning
df["label"] = 0.0

# KEEP ONLY what training needs
df = df[["text", "label"]]

df.head()


,text,label
0,Drug name: Lepirudin. Description: Lepirudin i...,0.0
1,Drug name: Cetuximab. Description: Cetuximab i...,0.0
2,Drug name: Dornase alfa. Description: Dornase ...,0.0
3,Drug name: Denileukin diftitox. Description: A...,0.0
4,Drug name: Etanercept. Description: Dimeric fu...,0.0


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset


In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

# Remove index column added by pandas
train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset   = val_dataset.remove_columns(["__index_level_0__"])

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)


Map:   0%|          | 0/8449 [00:00<?, ? examples/s]

Map:   0%|          | 0/2113 [00:00<?, ? examples/s]

In [ ]:

import os
import numpy as np
import pandas as pd

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from huggingface_hub import login

output_dir = "/content/pubmedbert_single_drug"  # TEMPORARY storage (Colab VM)

training_args = TrainingArguments(
    output_dir=output_dir,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,                 # 🔥 only BEST checkpoint kept

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,

    logging_dir=f"{output_dir}/logs",
    report_to="none",

    push_to_hub=True,                   # 🔥 push to Hugging Face
    hub_model_id="shubhangiisankar/pubmedbert-single-drug",
    hub_strategy="end"                  # 🔥 push ONLY best model at end
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.002688,0.000009
2,0.000955,0.000350
3,0.000513,0.000002


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3171, training_loss=0.0024571500528276025, metrics={'train_runtime': 702.9048, 'train_samples_per_second': 36.06, 'train_steps_per_second': 4.511, 'total_flos': 1667254010317056.0, 'train_loss': 0.0024571500528276025, 'epoch': 3.0})

In [ ]:
trainer.push_to_hub()


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...le_drug/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...le_drug/training_args.bin:   2%|1         |  81.0B / 5.26kB            

CommitInfo(commit_url='https://huggingface.co/shubhangiisankar/pubmedbert-single-drug/commit/d0fcb105ba15d0392da2e2eaf22d7360a53e3c19', commit_message='End of training', commit_description='', oid='d0fcb105ba15d0392da2e2eaf22d7360a53e3c19', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shubhangiisankar/pubmedbert-single-drug', endpoint='https://huggingface.co', repo_type='model', repo_id='shubhangiisankar/pubmedbert-single-drug'), pr_revision=None, pr_num=None)

In [ ]:
#LOADING THE MODEL FROM HUGGIG FACE HUB
model_id = "shubhangiisankar/pubmedbert-single-drug"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)
model.eval()

config.json:   0%|          | 0.00/800 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: shubhangiisankar/pubmedbert-single-drug
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_model.to(device)


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
def build_drug_text(row):
    return (
        f"Drug name: {row['name']}. "
        f"Description: {row['description']}. "
        f"Known side effects include: {row['side_effects']}."
    )


In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/drug_with_unique_side_effects (1).xlsx"
original_df = pd.read_excel(data_path)
original_df.columns = original_df.columns.str.strip().str.lower()


In [ ]:
def build_drug_text(row):
    return (
        f"Drug name: {row['name']}. "
        f"Description: {row['description']}. "
        f"Known side effects include: {row['side_effects']}."
    )

def get_drug_embedding(text):
    inputs = embed_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = embed_model(**inputs)

    # CLS token embedding
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd # Import pandas here for original_df

# Re-define original_df as it was not found in the kernel state
data_path = "/content/drive/MyDrive/drug_with_unique_side_effects (1).xlsx"
original_df = pd.read_excel(data_path)
original_df.columns = original_df.columns.str.strip().str.lower()

texts = original_df.apply(build_drug_text, axis=1).tolist()

def get_embeddings_batch(texts, batch_size=16):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]

        inputs = embed_tokenizer(
             batch_texts,
             return_tensors="pt",
             padding=True,
             truncation=True,
             max_length=128
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = embed_model(**inputs)

        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

    return np.vstack(all_embeddings)

embeddings = get_embeddings_batch(texts, batch_size=16)

  0%|          | 0/661 [00:00<?, ?it/s]


NameError: name 'embed_tokenizer' is not defined

In [ ]:
import pandas as pd

emb_df = pd.DataFrame(embeddings)
emb_df["name"] = original_df["name"].values


In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_pandas(emb_df)


/usr/local/lib/python3.12/dist-packages/datasets/table.py:719: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))


In [ ]:
hf_dataset.push_to_hub("shubhangiisankar/drug-embeddings")


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  30%|##9       | 2.68MB / 9.06MB            

CommitInfo(commit_url='https://huggingface.co/datasets/shubhangiisankar/drug-embeddings/commit/c3acd50116f8a81fdf8b1495a27466515437293c', commit_message='Upload dataset', commit_description='', oid='c3acd50116f8a81fdf8b1495a27466515437293c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/shubhangiisankar/drug-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='shubhangiisankar/drug-embeddings'), pr_revision=None, pr_num=None)

In [ ]:
import pandas as pd

emb_df = pd.DataFrame(
    embeddings,
    index=original_df["name"],   # optional but VERY useful
)

emb_df.head()


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
name,,,,,,,,,,,,,,,,,,,,,
Lepirudin,-0.086706,-0.272822,0.006013,0.178084,0.835426,0.057430,-0.434234,-0.011055,-0.332688,-0.080484,...,-0.332102,0.949730,-0.386424,0.057144,0.521282,-0.203681,-0.122927,0.076424,0.233346,-0.437812
Cetuximab,-0.094259,-0.283067,0.017644,0.161377,0.848129,0.055352,-0.429814,-0.008151,-0.329627,-0.084264,...,-0.329909,0.943212,-0.381336,0.057584,0.520653,-0.204906,-0.112365,0.077194,0.228499,-0.441774
Dornase alfa,-0.094259,-0.283067,0.017644,0.161377,0.848129,0.055352,-0.429814,-0.008151,-0.329627,-0.084264,...,-0.329909,0.943212,-0.381336,0.057584,0.520653,-0.204906,-0.112365,0.077194,0.228499,-0.441774
Denileukin diftitox,-0.084318,-0.290922,0.016451,0.159618,0.835346,0.066067,-0.417274,-0.000444,-0.340676,-0.085356,...,-0.330356,0.928493,-0.388264,0.034196,0.491320,-0.193629,-0.113132,0.066787,0.236671,-0.444165
Etanercept,-0.094259,-0.283067,0.017644,0.161377,0.848129,0.055352,-0.429814,-0.008151,-0.329627,-0.084264,...,-0.329909,0.943212,-0.381336,0.057584,0.520653,-0.204906,-0.112365,0.077194,0.228499,-0.441774


In [ ]:
emb_df.to_parquet("drug_embeddings.parquet")

In [ ]:
emb_df.shape


(10562, 768)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("shubhangiisankar/drug-embeddings")
emb_df = dataset["train"].to_pandas()

drug_names = emb_df["name"].tolist()
emb_matrix = emb_df.drop(columns=["name"]).values


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.06M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10562 [00:00<?, ? examples/s]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(emb_matrix)


In [ ]:
def infer_interaction(drug1, drug2):
    idx1 = drug_names.index(drug1)
    idx2 = drug_names.index(drug2)

    similarity_score = similarity_matrix[idx1][idx2]

    return similarity_score


In [ ]:
infer_interaction("Etanercept", "Adalimumab")


np.float32(0.9999741)

In [ ]:
import numpy as np

# Remove diagonal (self similarity)
sim_values = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

print("Max similarity:", sim_values.max())
print("Min similarity:", sim_values.min())
print("Mean similarity:", sim_values.mean())


Max similarity: 1.0000017
Min similarity: 0.999318
Mean similarity: 0.9999481


In [ ]:
from datasets import load_dataset

dataset = load_dataset("shubhangiisankar/drug-embeddings")
emb_df = dataset["train"].to_pandas()

embeddings = emb_df.drop(columns=["name"]).values


In [ ]:
import numpy as np
print(np.std(embeddings))


1.6909521


In [ ]:
print(embeddings[:5])


[[-0.08670614 -0.27282178  0.00601301 ...  0.07642375  0.23334596
  -0.4378124 ]
 [-0.09425941 -0.28306714  0.01764386 ...  0.07719444  0.22849926
  -0.44177374]
 [-0.09425941 -0.28306714  0.01764386 ...  0.07719444  0.22849926
  -0.44177374]
 [-0.08431844 -0.29092154  0.0164514  ...  0.06678707  0.23667069
  -0.4441653 ]
 [-0.09425941 -0.28306714  0.01764386 ...  0.07719444  0.22849926
  -0.44177374]]


In [ ]:
from sklearn.preprocessing import normalize

embeddings = normalize(embeddings)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)


In [ ]:
similarity_matrix[0, 1]


np.float32(0.9999848)

In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/drug_with_unique_side_effects (1).xlsx"
original_df = pd.read_excel(data_path)
original_df.columns = original_df.columns.str.strip().str.lower()


In [ ]:
from datasets import load_dataset

dataset = load_dataset("shubhangiisankar/drug-embeddings")

print(dataset)
print(dataset["train"].column_names)


DatasetDict({
    train: Dataset({
        features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150',

In [ ]:
from datasets import load_dataset
import numpy as np

dataset = load_dataset("shubhangiisankar/drug-embeddings")

df = dataset["train"].to_pandas()
embeddings = df.values

print(embeddings.shape)


(10562, 769)


In [ ]:
print(df.columns[:10])


Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype='object')


In [ ]:
print(len(df.columns))

769


In [ ]:
print(df.head())


          0         1         2         3         4         5         6  \
0 -0.086706 -0.272822  0.006013  0.178084  0.835426  0.057430 -0.434234   
1 -0.094259 -0.283067  0.017644  0.161377  0.848129  0.055352 -0.429814   
2 -0.094259 -0.283067  0.017644  0.161377  0.848129  0.055352 -0.429814   
3 -0.084318 -0.290922  0.016451  0.159618  0.835346  0.066067 -0.417274   
4 -0.094259 -0.283067  0.017644  0.161377  0.848129  0.055352 -0.429814   

          7         8         9  ...       759       760       761       762  \
0 -0.011055 -0.332688 -0.080484  ...  0.949730 -0.386424  0.057144  0.521282   
1 -0.008151 -0.329627 -0.084264  ...  0.943212 -0.381336  0.057584  0.520653   
2 -0.008151 -0.329627 -0.084264  ...  0.943212 -0.381336  0.057584  0.520653   
3 -0.000444 -0.340676 -0.085356  ...  0.928493 -0.388264  0.034196  0.491320   
4 -0.008151 -0.329627 -0.084264  ...  0.943212 -0.381336  0.057584  0.520653   

        763       764       765       766       767                 

In [ ]:
embeddings = df.to_numpy()


In [ ]:
print(embeddings.shape)


(10562, 769)


In [ ]:
embeddings = df.iloc[:, :768].values
print(embeddings.shape)


(10562, 768)


In [ ]:
from sklearn.preprocessing import normalize

embeddings = normalize(embeddings)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)


In [ ]:
import numpy as np

# Remove diagonal (self similarity)
sim_values = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

print("Max similarity:", sim_values.max())
print("Min similarity:", sim_values.min())
print("Mean similarity:", sim_values.mean())


Max similarity: 1.0000007
Min similarity: 0.99931747
Mean similarity: 0.99994797


In [ ]:
drug_to_effects = {}

for _, row in original_df.iterrows():
    name = row["name"]
    effects = str(row["side_effects"]).lower()
    effects_list = [e.strip() for e in effects.split(",")]
    drug_to_effects[name] = set(effects_list)


In [ ]:
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

embeddings = normalize(embeddings)
similarity_matrix = cosine_similarity(embeddings)

In [ ]:
name_to_index = {
    name: idx
    for idx, name in enumerate(original_df["name"])
}


In [ ]:
def compute_risk(drug1, drug2):

    idx1 = name_to_index[drug1]
    idx2 = name_to_index[drug2]

    emb_score = similarity_matrix[idx1, idx2]

    se1 = drug_to_effects[drug1]
    se2 = drug_to_effects[drug2]

    common = se1.intersection(se2)

    overlap_ratio = len(common) / max(len(se1.union(se2)), 1)

    final_score = 0.6 * emb_score + 0.4 * overlap_ratio

    return emb_score, overlap_ratio, final_score, common


In [ ]:
def generate_interaction_report(drug1, drug2):

    emb_score, overlap_ratio, final_score, common = compute_risk(drug1, drug2)

    if final_score > 0.75:
        level = "🔴 High Interaction Risk"
    elif final_score > 0.5:
        level = "🟡 Moderate Interaction Risk"
    else:
        level = "🟢 Low Interaction Risk"

    if len(common) > 0:
        common_text = ", ".join(common)
        explanation = (
            f"{drug1} and {drug2} share overlapping side effects such as "
            f"{common_text}. Combined usage may increase adverse reaction risk."
        )
    else:
        explanation = (
            f"No direct side-effect overlap detected between "
            f"{drug1} and {drug2}. Interaction risk is inferred primarily "
            f"from pharmacological similarity."
        )

    return {
        "Risk Level": level,
        "Embedding Similarity": round(float(emb_score), 3),
        "Side Effect Overlap": round(float(overlap_ratio), 3),
        "Final Risk Score": round(float(final_score), 3),
        "Explanation": explanation
    }


In [ ]:
generate_interaction_report("Reteplase", "Asparaginase")


{'Risk Level': '🟡 Moderate Interaction Risk',
 'Embedding Similarity': 1.0,
 'Side Effect Overlap': 0.115,
 'Final Risk Score': 0.646,
 'Explanation': 'Reteplase and Asparaginase share overlapping side effects such as shortness of breath, allergic reactions, weight loss. Combined usage may increase adverse reaction risk.'}

In [ ]:
def improved_interaction(drug1, drug2, threshold=0.5):
    idx1 = name_to_index[drug1]
    idx2 = name_to_index[drug2]

    embedding_score = similarity_matrix[idx1, idx2]

    effects1 = set(original_df.loc[idx1, "side_effects"].lower().split(","))
    effects2 = set(original_df.loc[idx2, "side_effects"].lower().split(","))

    overlap = effects1.intersection(effects2)
    overlap_ratio = len(overlap) / max(len(effects1), 1)

    final_score = 0.6 * embedding_score + 0.4 * overlap_ratio

    if final_score > 0.75:
        risk = "High Interaction Risk"
    elif final_score > 0.5:
        risk = "Moderate Interaction Risk"
    else:
        risk = "Low Interaction Risk"

    return {
        "drug1": drug1,
        "drug2": drug2,
        "embedding_score": round(float(embedding_score), 4),
        "side_effect_overlap": round(float(overlap_ratio), 4),
        "matched_effects": list(overlap),
        "risk": risk
    }


In [ ]:
def generate_explanation(result):
    drug1 = result["drug1"]
    drug2 = result["drug2"]
    risk = result["risk"]
    matched = result["matched_effects"]

    if matched:
        effects_text = ", ".join(matched)
        explanation = (
            f"{drug1} and {drug2} share overlapping side effects including "
            f"{effects_text}. This suggests a {risk.lower()}. "
            "Clinical monitoring is recommended when these drugs are co-administered."
        )
    else:
        explanation = (
            f"{drug1} and {drug2} do not show significant overlap in side effects. "
            f"The estimated risk level is {risk.lower()}. "
            "However, clinical evaluation is still advised."
        )

    return explanation


In [ ]:
def check_interaction(drug1, drug2):
    result = improved_interaction(drug1, drug2)
    return generate_explanation(result)


In [ ]:
print("1st drug\n")
print(check_interaction("Cetuximab", "Trastuzumab"))
print("2nd drug\n")
print(check_interaction("Spermine", "Ibritumomab tiuxetan"))
print("3rd drug \n")
print(check_interaction("L-Methionine", "Glutathione"))


1st drug

Cetuximab and Trastuzumab share overlapping side effects including  shortness of breath,  indigestion,  palpitations,  flushing. This suggests a moderate interaction risk. Clinical monitoring is recommended when these drugs are co-administered.
2nd drug

Spermine and Ibritumomab tiuxetan share overlapping side effects including  gastric irritation,  constipation,  loss of appetite,  insomnia. This suggests a high interaction risk. Clinical monitoring is recommended when these drugs are co-administered.
3rd drug 

L-Methionine and Glutathione share overlapping side effects including  anemia,  diarrhea,  vomiting,  blurred vision. This suggests a moderate interaction risk. Clinical monitoring is recommended when these drugs are co-administered.


In [ ]:
test = improved_interaction("Etanercept", "Adalimumab")
print(test)


{'drug1': 'Etanercept', 'drug2': 'Adalimumab', 'embedding_score': 1.0, 'side_effect_overlap': 0.1818, 'matched_effects': [' weakness', ' weight gain'], 'risk': 'Moderate Interaction Risk'}


In [ ]:
print("original_df exists:", "original_df" in globals())
print("embeddings exists:", "embeddings" in globals())
print("similarity_matrix exists:", "similarity_matrix" in globals())
print("name_to_index exists:", "name_to_index" in globals())
print("improved_interaction exists:", "improved_interaction" in globals())


original_df exists: True
embeddings exists: True
similarity_matrix exists: True
name_to_index exists: True
improved_interaction exists: True


In [ ]:
# Clean drug names properly
original_df["name"] = original_df["name"].str.strip().str.lower()

# Rebuild mapping
name_to_index = {name: idx for idx, name in enumerate(original_df["name"])}

print("Mapping rebuilt. Total drugs:", len(name_to_index))


Mapping rebuilt. Total drugs: 10562


In [ ]:
def improved_interaction(drug1, drug2, threshold=0.5):
    drug1 = drug1.strip().lower()
    drug2 = drug2.strip().lower()

    if drug1 not in name_to_index or drug2 not in name_to_index:
        return {"error": "One or both drugs not found in dataset."}

    idx1 = name_to_index[drug1]
    idx2 = name_to_index[drug2]

    embedding_score = similarity_matrix[idx1, idx2]

    effects1 = set(original_df.loc[idx1, "side_effects"].lower().split(","))
    effects2 = set(original_df.loc[idx2, "side_effects"].lower().split(","))

    overlap = effects1.intersection(effects2)
    overlap_score = len(overlap) / max(len(effects1), len(effects2))

    final_score = 0.7 * embedding_score + 0.3 * overlap_score

    if final_score > 0.75:
        risk = "High Interaction Risk"
    elif final_score > 0.5:
        risk = "Moderate Interaction Risk"
    else:
        risk = "Low Interaction Risk"

    return {
        "drug1": drug1,
        "drug2": drug2,
        "embedding_score": round(float(embedding_score), 4),
        "side_effect_overlap": round(float(overlap_score), 4),
        "matched_effects": list(overlap),
        "final_score": round(float(final_score), 4),
        "risk": risk
    }


In [ ]:
print(improved_interaction("Etanercept", "Adalimumab"))


{'drug1': 'etanercept', 'drug2': 'adalimumab', 'embedding_score': 1.0, 'side_effect_overlap': 0.125, 'matched_effects': [' weakness', ' weight gain'], 'final_score': 0.7375, 'risk': 'Moderate Interaction Risk'}


In [ ]:
d1 = "etanercept"
d2 = "adalimumab"

idx1 = name_to_index[d1]
idx2 = name_to_index[d2]

print("Drug 1 Side Effects:\n", original_df.loc[idx1, "side_effects"])
print("\nDrug 2 Side Effects:\n", original_df.loc[idx2, "side_effects"])


Drug 1 Side Effects:
 Headache, Hives, Weakness, Weight gain, Shortness of breath, Dizziness, Cough, Muscle pain, Insomnia, Gastric irritation, Anemia

Drug 2 Side Effects:
 High blood pressure, Vomiting, Tremor, Weakness, Rash, Numbness, Liver enzyme elevation, Back pain, Blurred vision, Increased heart rate, Chills, Weight gain, Fever, Low blood pressure, Diarrhea, Palpitations


In [ ]:
# Curated validation pairs based on known public drug interaction knowledge
external_reference = {
    ("aspirin", "warfarin"): "High Interaction Risk",
    ("ibuprofen", "warfarin"): "High Interaction Risk",
    ("lisinopril", "spironolactone"): "High Interaction Risk",
    ("metformin", "cetirizine"): "Low Interaction Risk",
    ("paracetamol", "amoxicillin"): "Low Interaction Risk"
}


In [ ]:
def validate_against_external(drug1, drug2):
    drug1 = drug1.lower()
    drug2 = drug2.lower()

    model_result = improved_interaction(drug1, drug2)

    ref = external_reference.get((drug1, drug2)) or \
          external_reference.get((drug2, drug1))

    if ref:
        match = model_result["risk"] == ref
    else:
        match = "No external reference"

    return {
        "drug1": drug1,
        "drug2": drug2,
        "model_prediction": model_result["risk"],
        "external_reference": ref,
        "match": match
    }


In [ ]:
for pair in external_reference.keys():
    print(validate_against_external(pair[0], pair[1]))


KeyError: 'risk'

In [ ]:
test = improved_interaction("etanercept", "adalimumab")
print(test.keys())


dict_keys(['drug1', 'drug2', 'embedding_score', 'side_effect_overlap', 'matched_effects', 'final_score', 'risk'])


In [ ]:
print(improved_interaction("etancercept", "adalimumab"))


{'error': 'One or both drugs not found in dataset.'}


In [ ]:
def improved_interaction(drug1, drug2, threshold=0.5):
    drug1 = drug1.lower()
    drug2 = drug2.lower()

    idx1 = name_to_index[drug1]
    idx2 = name_to_index[drug2]

    embedding_score = similarity_matrix[idx1, idx2]

    effects1 = set(original_df.loc[idx1, "side_effects"].lower().split(","))
    effects2 = set(original_df.loc[idx2, "side_effects"].lower().split(","))

    overlap = effects1.intersection(effects2)
    overlap_score = len(overlap) / max(len(effects1), 1)

    final_score = 0.6 * embedding_score + 0.4 * overlap_score

    if final_score > 0.75:
        risk = "High Interaction Risk"
    elif final_score > 0.5:
        risk = "Moderate Interaction Risk"
    else:
        risk = "Low Interaction Risk"

    return {
        "drug1": drug1,
        "drug2": drug2,
        "embedding_score": float(embedding_score),
        "side_effect_overlap": float(overlap_score),
        "matched_effects": list(overlap),
        "final_score": float(final_score),
        "risk": risk
    }


In [ ]:
print(improved_interaction("Bivalirudin", "adalimumab"))


{'drug1': 'bivalirudin', 'drug2': 'adalimumab', 'embedding_score': 0.9999739527702332, 'side_effect_overlap': 0.375, 'matched_effects': [' increased heart rate', ' diarrhea', ' palpitations', ' tremor', ' blurred vision', ' numbness'], 'final_score': 0.7499843835830688, 'risk': 'Moderate Interaction Risk'}


In [ ]:
for pair in external_reference.keys():
    print(validate_against_external(pair[0], pair[1]))


KeyError: 'aspirin'

In [ ]:
def improved_interaction(drug1, drug2, threshold=0.5):

    drug1 = drug1.lower().strip()
    drug2 = drug2.lower().strip()

    idx1 = name_to_index[drug1]
    idx2 = name_to_index[drug2]

    embedding_score = similarity_matrix[idx1, idx2]

    effects1 = set(original_df.loc[idx1, "side_effects"].lower().split(","))
    effects2 = set(original_df.loc[idx2, "side_effects"].lower().split(","))

    overlap = effects1.intersection(effects2)
    overlap_ratio = len(overlap) / max(len(effects1), 1)

    final_score = 0.6 * embedding_score + 0.4 * overlap_ratio

    if final_score > 0.75:
        risk = "High Interaction Risk"
    elif final_score > 0.5:
        risk = "Moderate Interaction Risk"
    else:
        risk = "Low Interaction Risk"

    return {
        "drug1": drug1,
        "drug2": drug2,
        "embedding_score": float(embedding_score),
        "side_effect_overlap": float(overlap_ratio),
        "matched_effects": list(overlap),
        "final_score": float(final_score),
        "risk": risk
    }


In [ ]:
model_result = improved_interaction(drug1, drug2)

if "error" in model_result:
    return model_result


NameError: name 'drug1' is not defined

In [ ]:
def safe_validate(drug1, drug2):
    drug1 = drug1.lower().strip()
    drug2 = drug2.lower().strip()

    if drug1 not in name_to_index:
        return {"error": f"{drug1} not found in dataset"}

    if drug2 not in name_to_index:
        return {"error": f"{drug2} not found in dataset"}

    return improved_interaction(drug1, drug2)


In [ ]:
print("Total drugs available:", len(name_to_index))


Total drugs available: 10562


In [ ]:
drug_name = "aspirin".lower()
if drug_name in name_to_index:
    print("Drug exists in dataset")
else:
    print("Drug NOT present in dataset")



Drug NOT present in dataset


In [ ]:
print(original_df["name"].head(20))


0                 lepirudin
1                 cetuximab
2              dornase alfa
3       denileukin diftitox
4                etanercept
5               bivalirudin
6                leuprolide
7     peginterferon alfa-2a
8                 alteplase
9                sermorelin
10       interferon alfa-n1
11         darbepoetin alfa
12                urokinase
13                goserelin
14                reteplase
15           erythropoietin
16        salmon calcitonin
17       interferon alfa-n3
18            pegfilgrastim
19             sargramostim
Name: name, dtype: object


In [ ]:
print(original_df["name"].str.contains("aspirin", case=False).sum())


2


In [ ]:
print(safe_validate("etanercept", "adalimumab"))


{'drug1': 'etanercept', 'drug2': 'adalimumab', 'embedding_score': 0.9999739527702332, 'side_effect_overlap': 0.18181818181818182, 'matched_effects': [' weakness', ' weight gain'], 'final_score': 0.6727116703987122, 'risk': 'Moderate Interaction Risk'}


In [ ]:
result = improved_interaction("etanercept", "adalimumab")
print(result)


{'drug1': 'etanercept', 'drug2': 'adalimumab', 'embedding_score': 0.9999739527702332, 'side_effect_overlap': 0.18181818181818182, 'matched_effects': [' weakness', ' weight gain'], 'final_score': 0.6727116703987122, 'risk': 'Moderate Interaction Risk'}


In [ ]:
print("original_df exists:", "original_df" in globals())
print("name_to_index exists:", "name_to_index" in globals())
print("similarity_matrix exists:", "similarity_matrix" in globals())
print("improved_interaction exists:", "improved_interaction" in globals())
print("ddi_df exists:", "ddi_df" in globals())


original_df exists: True
name_to_index exists: True
similarity_matrix exists: True
improved_interaction exists: True
ddi_df exists: False


In [ ]:
import pandas as pd

# Update this path if needed
csv_path = "db_drug_interactions.csv"

ddi_df = pd.read_csv(csv_path)

print("DDI dataset loaded:", ddi_df.shape)
print("Columns:", ddi_df.columns.tolist())
print(ddi_df.head())

DDI dataset loaded: (191541, 3)
Columns: ['Drug 1', 'Drug 2', 'Interaction Description']
                Drug 1       Drug 2  \
0           Trioxsalen  Verteporfin   
1  Aminolevulinic acid  Verteporfin   
2     Titanium dioxide  Verteporfin   
3     Tiaprofenic acid  Verteporfin   
4          Cyamemazine  Verteporfin   

                             Interaction Description  
0  Trioxsalen may increase the photosensitizing a...  
1  Aminolevulinic acid may increase the photosens...  
2  Titanium dioxide may increase the photosensiti...  
3  Tiaprofenic acid may increase the photosensiti...  
4  Cyamemazine may increase the photosensitizing ...  


In [ ]:
import pandas as pd
csv_path = "db_drug_interactions.csv"

ddi_df = pd.read_csv(csv_path)
ddi_df["Drug 1"] = ddi_df["Drug 1"].str.lower().str.strip()
ddi_df["Drug 2"] = ddi_df["Drug 2"].str.lower().str.strip()

print("Normalization complete")


FileNotFoundError: [Errno 2] No such file or directory: 'db_drug_interactions.csv'